In [418]:
import pandas as pd
from collections import Counter
import numpy as np
from pathlib import Path
from scipy.stats import entropy  # entropy is a function to compute KL-divergence


In [419]:
#Define QID - question matching
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing", 
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}

def get_counts(path, qids): #function to get a dict with counts for each question
    value_list = []

    for id in qids:
        df = pd.read_csv(path + id + ".csv")
        df['Response'] = pd.to_numeric(df['Response'], errors='coerce') 
        value_counts = Counter(df['Response'].dropna())

        if id in ["V201324", "V202332"]:
            answers = {i: value_counts[i] for i in range(1, 6)}
        else:
            answers = {i: value_counts[i] for i in range(1, 4)}

        value_list.append({name_dict[id]: answers})

    return value_list


### Select model

In [283]:
# LLama8b
model_name = 'llama8b'
replicate_path = "MLLM Results/main_mq_results/full_results_2020_"
reformulated_path = "MLLM Results/main_mq_reform_3rdP_results/full_results_2020_"
priming_path = "MLLM Results/Priming/Original/full_results_2020_"
preamble_path = "MLLM Results/Preamble/Original/full_results_2020_"
reversed_path = "MLLM Results/main_mq_reverseV2_results/full_results_2020_"




In [298]:
# LLama70b
model_name = 'llama70b'
replicate_path = "Publication Results/70B-Llama/main_mq_results/full_results_2020_"
reformulated_path = "Publication Results/70B-Llama/main_mq_reform_results/full_results_2020_"
priming_path = "Publication Results/70B-Llama/main_mq_priming_results/full_results_2020_"
preamble_path = "Publication Results/70B-Llama/main_mq_preamble_results/full_results_2020_"
reversed_path = "Publication Results/70B-Llama/main_mq_reverseV2_results/full_results_2020_"
 


In [ ]:
# GPT-4.1
model_name = 'gpt4.1'
replicate_path = "Publication Results/GPT4.1/main_mq_results/full_results_2020_"
reformulated_path = "Publication Results/GPT4.1/main_mq_reform_results/full_results_2020_"
priming_path = "Publication Results/GPT4.1/main_mq_priming_results/full_results_2020_"
preamble_path = "Publication Results/GPT4.1/main_mq_preamble_results/full_results_2020_"
reversed_path = "Publication Results/GPT4.1/main_mq_reverseV2_results/full_results_2020_"

replicate_path = "Publication Results/Gpt4.1-Mini-Determinstic/main_mq_results/full_results_2020_"
reformulated_path = "Publication Results/Gpt4.1-Mini-Determinstic/main_mq_reform_results/full_results_2020_"
priming_path = "Publication Results/Gpt4.1-Mini-Determinstic/main_mq_priming_results/full_results_2020_"
preamble_path = "Publication Results/Gpt4.1-Mini-Determinstic/main_mq_preamble_results/full_results_2020_"
reversed_path = "Publication Results/Gpt4.1-Mini-Determinstic/main_mq_reverseV2_results/full_results_2020_"

 


In [421]:
original_results = get_counts("results/main_mq_results_unzipped/full_results_2020_", qids)

replicate_results = get_counts(replicate_path, qids)
reformulated_results = get_counts(reformulated_path, qids)
priming_results = get_counts(priming_path, qids)
preamble_results = get_counts(preamble_path, qids)

#Only some IDs were used for reverse-coded
qids_reverse_coded = ["V201416", "V202234", "V202257", "V202287", "V202332", "V202371"]
reversed_results = get_counts(reversed_path, qids_reverse_coded)

In [422]:
def get_human_counts(path, qids): #getting results from anes 2020
     anes_df = pd.read_csv(path)
     human_values = []

     for id in qids:
          human_counts = Counter(anes_df[id].dropna())
          if id in ["V201324", "V202332"]:
               answers = {i: human_counts[i] for i in range(1, 6)}
          else:
               answers = {i: human_counts[i] for i in range(1, 4)}

          human_values.append({name_dict[id]: answers})

     return human_values

human_results = get_human_counts("data/2020 ANES_test.csv", qids)

/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_57581/1072730952.py:2: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_df = pd.read_csv(path)


In [423]:
from copy import deepcopy 

reversed_results = {list(d.keys())[0]: list(d.values())[0] for d in reversed_results}
reversed_results_copy = deepcopy(reversed_results)

reversed_results["Gay Marriage"][1] = reversed_results_copy["Gay Marriage"][3]
reversed_results["Gay Marriage"][3] = reversed_results_copy["Gay Marriage"][1]

reversed_results["Refugee Allowing"][1] = reversed_results_copy["Refugee Allowing"][2]
reversed_results["Refugee Allowing"][2] = reversed_results_copy["Refugee Allowing"][1]

reversed_results["Income Inequality"][1] = reversed_results_copy["Income Inequality"][2]
reversed_results["Income Inequality"][2] = reversed_results_copy["Income Inequality"][1]

reversed_results["Gender Role"][1] = reversed_results_copy["Gender Role"][2]
reversed_results["Gender Role"][2] = reversed_results_copy["Gender Role"][1]

reversed_results["Climate Change"][1] = reversed_results_copy["Climate Change"][5]
reversed_results["Climate Change"][2] = reversed_results_copy["Climate Change"][4]
reversed_results["Climate Change"][4] = reversed_results_copy["Climate Change"][2]
reversed_results["Climate Change"][5] = reversed_results_copy["Climate Change"][1]

reversed_results["Race Diversity"][1] = reversed_results_copy["Race Diversity"][2]
reversed_results["Race Diversity"][2] = reversed_results_copy["Race Diversity"][1]

reversed_results = [{key: value} for key, value in reversed_results.items()]

In [424]:
#JSD divergence is a more numerically stable version of KL-divergence - better for our case
def jsd_divergence_between_sets(set1, set2, base=2):
    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [425]:
#Get all the results and divergences
jsd_results_previous_study = jsd_divergence_between_sets(human_results, original_results, base=2)

jsd_results_replicate = jsd_divergence_between_sets(human_results, replicate_results, base=2)
jsd_results_reformulated = jsd_divergence_between_sets(human_results, reformulated_results, base=2)
jsd_results_priming = jsd_divergence_between_sets(human_results, priming_results, base=2)
jsd_results_preamble = jsd_divergence_between_sets(human_results, preamble_results, base=2)
jsd_results_reversed = jsd_divergence_between_sets(human_results, reversed_results, base = 2)


diff_replicate_vs_previous = {key: (jsd_results_replicate[key] - jsd_results_previous_study[key]) for key in jsd_results_replicate.keys()}
diff_reformulated_vs_replicate = {key: (jsd_results_reformulated[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}
diff_priming_vs_replicate = {key: (jsd_results_priming[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}
diff_preamble_vs_replicate = {key: (jsd_results_preamble[key] - jsd_results_replicate[key]) for key in jsd_results_replicate.keys()}
diff_reversed_vs_replicate = {key: (jsd_results_reversed[key] - jsd_results_replicate[key]) for key in jsd_results_reversed.keys()}


In [428]:
# format data
# Difference in JS-divergence between replicate and 4 experimental conditions
results_mapping = [
    ('Replicate', jsd_results_replicate),
    ('Reformulated - Replicate', diff_reformulated_vs_replicate),
    ('Reverse-coded - Replicate', diff_reversed_vs_replicate),
    ('Priming - Replicate', diff_priming_vs_replicate),
    ('Preamble - Replicate', diff_preamble_vs_replicate),
]

question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']
sig_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    # Extract values from the dictionary in the precise 'question_order'
    ordered_values = [
        result_dict.get(q, np.nan)  # Use .get() and np.nan for missing keys
        for q in question_order
    ]
    # Add the column to the DataFrame, rounding to 3 decimals
    sig_df[column_name] = np.round(ordered_values, 3)

In [ ]:
# format data for JSD not JSD diff
results_mapping = [
    ('Replicate', jsd_results_replicate),
    ('Reformulated', jsd_results_reformulated),
    ('Reverse-coded', jsd_results_reversed),
    ('Priming', jsd_results_priming),
    ('Preamble', jsd_results_preamble)
]

question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']
sig_df2 = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    ordered_values = [
        result_dict.get(q, np.nan)  # Use .get() and np.nan for missing keys
        for q in question_order
    ]
    sig_df2[column_name] = np.round(ordered_values, 3)

In [432]:
# print(model_name)
# # sig_df2[['Replicate']].mean(),sig_df2[['Reformulated']].mean(),sig_df2[['Reverse-coded']].mean(),sig_df2[['Priming']].mean(),sig_df2[['Preamble']].mean()
# # check T=0

In [433]:
# sig_df2[['Replicate']].mean(),sig_df2[['Reformulated']].mean(),sig_df2[['Reverse-coded']].mean(),sig_df2[['Priming']].mean(),sig_df2[['Preamble']].mean()
# # check T=1

In [434]:
print(model_name)
sig_df

gpt4.1


,Question,Replicate,Reformulated - Replicate,Reverse-coded - Replicate,Priming - Replicate,Preamble - Replicate
0,Climate Change,0.058,-0.032,0.283,0.153,0.023
1,Current Economy,0.232,-0.060,NaN,0.048,-0.011
2,Refugee Allowing,0.113,-0.086,-0.064,-0.004,0.136
3,Health Insurance,0.045,0.026,NaN,0.034,0.162
4,Income Inequality,0.044,0.060,-0.040,0.042,0.046
5,Race Diversity,0.254,-0.068,-0.140,0.013,-0.006
6,Drug Addiction,0.166,-0.022,NaN,0.007,0.007
7,Gay Marriage,0.041,-0.009,0.054,-0.012,0.019
8,Gun Regulation,0.003,0.012,NaN,0.011,0.010
9,Gender Role,0.077,-0.067,0.770,0.106,0.063


In [435]:
file_path = f'tables/{model_name}-jsd-diff-t=0.csv'
sig_df.to_csv(file_path, index=False)

In [275]:
# #For table1
# display(jsd_results_replicate)          # Baseline
# display(jsd_results_reformulated)       # 1a (Re-form.)
# display(diff_reformulated_vs_replicate) # Δ (1a–Base)

# display(jsd_results_priming)       
# display(diff_priming_vs_replicate) 

# display(jsd_results_preamble)       
# display(diff_preamble_vs_replicate) 

# display(jsd_results_reversed)       
# display(diff_reversed_vs_replicate) 

In [437]:
def jsd_divergence_for_bootstrapping(human_sample, bootstrap_sample, base=2):
    p_counts = human_sample
    q_counts = bootstrap_sample

    p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
    q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

    p /= p.sum()
    q /= q.sum()

    epsilon = 1e-10
    p = np.clip(p, epsilon, 1)
    q = np.clip(q, epsilon, 1)

    m = 0.5 * (p + q)
    jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base))

    return jsd_pq


### CIs (Replicate vs conditions) [not used]

In [351]:
def bootstrap_jsd_replicate(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    responses = np.array(question_df['Response'].dropna())

    replicate_responses = {list(d.keys())[0]: list(d.values())[0] for d in replicate_results}[name_dict[qid]]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True) #sample with replacement
        
        counts = Counter(sample)

        jsd_val = jsd_divergence_for_bootstrapping(replicate_responses, counts, base=base) #compute JSD for this iteration
        jsd_samples.append(jsd_val) #append to the samples list

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples  # <— raw samples for delta CI

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])
    
    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))

def bootstrap_jsd_replicate_for_reverse_coded(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    reverse_mappings = {
        "Gay Marriage": {1: 3, 3: 1},
        "Refugee Allowing": {1: 2, 2: 1},
        "Income Inequality": {1: 2, 2: 1},
        "Gender Role": {1: 2, 2: 1},
        "Climate Change": {1: 5, 2: 4, 4: 2, 5: 1},
        "Race Diversity": {1: 2, 2: 1}
    }

    qname = name_dict[qid]
    responses = np.array(question_df['Response'].dropna())

    #apply the map to swap the answers
    if qname in reverse_mappings:
        mapping = reverse_mappings[qname]
        responses = np.array([mapping.get(val, val) for val in responses])

    replicate_responses = {list(d.keys())[0]: list(d.values())[0] for d in replicate_results}[name_dict[qid]]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True)
        counts = Counter(sample)
        jsd_val = jsd_divergence_for_bootstrapping(replicate_responses, counts, base=base)
        jsd_samples.append(jsd_val)

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])

    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))



In [278]:
#Create CIs (Replicate vs conditions) and compare them
ci_replicate = []
ci_reformulated = []
ci_preamble = []
ci_priming = []
ci_reverse_coded = []

for qid in qids:
    ci_replicate.append((name_dict[qid], bootstrap_jsd_replicate(path = replicate_path, qid = qid)))
    ci_reformulated.append((name_dict[qid], bootstrap_jsd_replicate(path = reformulated_path, qid = qid)))
    ci_priming.append((name_dict[qid], bootstrap_jsd_replicate(path = priming_path, qid = qid)))
    ci_preamble.append((name_dict[qid], bootstrap_jsd_replicate(path = preamble_path, qid = qid)))

ci_replicate = [{item[0] : item[1]} for item in ci_replicate]
ci_reformulated = [{item[0] : item[1]} for item in ci_reformulated]
ci_priming = [{item[0] : item[1]} for item in ci_priming]
ci_preamble = [{item[0] : item[1]} for item in ci_preamble]

ci_replicate = {list(d.keys())[0]: list(d.values())[0] for d in ci_replicate}
ci_priming = {list(d.keys())[0]: list(d.values())[0] for d in ci_priming}
ci_reformulated = {list(d.keys())[0]: list(d.values())[0] for d in ci_reformulated}
ci_preamble = {list(d.keys())[0]: list(d.values())[0] for d in ci_preamble}


ci_reverse_coded = []

for qid in qids_reverse_coded:
    ci_reverse_coded.append((name_dict[qid], bootstrap_jsd_replicate_for_reverse_coded(path = reversed_path, qid = qid)))

ci_reverse_coded = [{item[0] : item[1]} for item in ci_reverse_coded]
ci_reverse_coded = {list(d.keys())[0]: list(d.values())[0] for d in ci_reverse_coded}

In [279]:
# Mean JS divergence and 95% bootstrap confidence intervals between Replicate and silicon distributions by item and condition.
def format_ci_data(ci_tuple):
    # Convert the list of tuples structure into a dictionary for easy access
    if ci_tuple is None:
        return "N/A"
    data = dict(ci_tuple)
    
    mean = data['mean']
    ci_lower = data['ci_lower']
    ci_upper = data['ci_upper']

    # Round and format
    mean_str = f"{mean:.4f}"
    lower_str = f"{ci_lower:4f}"
    upper_str = f"{ci_upper:.4f}"

    return f"{mean_str} [{lower_str}, {upper_str}]"

results_mapping = [
    # ('Replicate', ci_replicate),
    ('Reformulated', ci_reformulated),
    ('Reverse-coded', ci_reverse_coded),
    ('Priming', ci_priming),
    ('Preamble', ci_preamble),
]
question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']

ci_df = pd.DataFrame({'Question': question_order})
for column_name, result_dict in results_mapping:
    ordered_formatted_values = [
        format_ci_data(result_dict.get(q, None))
        for q in question_order
    ]

    ci_df[column_name] = ordered_formatted_values


In [280]:
ci_df

,Question,Reformulated,Reverse-coded,Priming,Preamble
0,Climate Change,"0.0750 [0.069125, 0.0811]","0.3679 [0.361380, 0.3742]","0.1087 [0.103022, 0.1146]","0.0148 [0.012203, 0.0177]"
1,Current Economy,"0.0732 [0.067545, 0.0789]",N/A,"0.0820 [0.075865, 0.0883]","0.0082 [0.006218, 0.0103]"
2,Refugee Allowing,"0.0247 [0.021355, 0.0283]","0.0102 [0.008029, 0.0127]","0.0148 [0.012193, 0.0176]","0.0464 [0.041999, 0.0508]"
3,Health Insurance,"0.0046 [0.003118, 0.0061]",N/A,"0.0095 [0.007448, 0.0117]","0.0520 [0.047233, 0.0568]"
4,Income Inequality,"0.0534 [0.048633, 0.0583]","0.0240 [0.020677, 0.0276]","0.0071 [0.005281, 0.0090]","0.0106 [0.008386, 0.0129]"
5,Race Diversity,"0.0754 [0.070150, 0.0805]","0.0518 [0.047037, 0.0566]","0.0065 [0.005622, 0.0070]","0.0004 [0.000071, 0.0009]"
6,Drug Addiction,"0.0243 [0.021116, 0.0275]",N/A,"0.0022 [0.002209, 0.0022]","0.0022 [0.002209, 0.0022]"
7,Gay Marriage,"0.0005 [0.000114, 0.0010]","0.0286 [0.026920, 0.0303]","0.0087 [0.006829, 0.0109]","0.0061 [0.004472, 0.0080]"
8,Gun Regulation,"0.0181 [0.015189, 0.0211]",N/A,"0.0047 [0.003339, 0.0064]","0.0047 [0.003258, 0.0063]"
9,Gender Role,"0.0740 [0.068248, 0.0801]","0.9735 [0.971303, 0.9746]","0.0380 [0.034798, 0.0414]","0.0075 [0.005692, 0.0094]"


In [ ]:

file_path = f'tables/{model_name}-jsd-ci-replicate.csv'
ci_df.to_csv(file_path, index=False)

### CIs (Human vs conditions)

In [438]:
human_df = pd.read_csv("data/2020 ANES_test.csv")

def bootstrap_jsd(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    responses = np.array(question_df['Response'].dropna())

    human_responses = {list(d.keys())[0]: list(d.values())[0] for d in human_results}[name_dict[qid]]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True) #sample with replacement
        
        counts = Counter(sample)

        jsd_val = jsd_divergence_for_bootstrapping(human_responses, counts, base=base) #compute JSD for this iteration
        jsd_samples.append(jsd_val) #append to the samples list

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples  # <— raw samples for delta CI

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])
    
    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))

def bootstrap_jsd_for_reverse_coded(path, qid, n_boot=2000, base=2, return_samples=False):

    question_df = pd.read_csv(path + qid + ".csv")
    question_df['Response'] = pd.to_numeric(question_df['Response'], errors='coerce')

    reverse_mappings = {
        "Gay Marriage": {1: 3, 3: 1},
        "Refugee Allowing": {1: 2, 2: 1},
        "Income Inequality": {1: 2, 2: 1},
        "Gender Role": {1: 2, 2: 1},
        "Climate Change": {1: 5, 2: 4, 4: 2, 5: 1},
        "Race Diversity": {1: 2, 2: 1}
    }

    qname = name_dict[qid]
    responses = np.array(question_df['Response'].dropna())

    #apply the map to swap the answers
    if qname in reverse_mappings:
        mapping = reverse_mappings[qname]
        responses = np.array([mapping.get(val, val) for val in responses])

    human_responses = {list(d.keys())[0]: list(d.values())[0] for d in human_results}[qname]

    jsd_samples = []

    for _ in range(n_boot):
        sample = np.random.choice(responses, size=len(responses), replace=True)
        counts = Counter(sample)
        jsd_val = jsd_divergence_for_bootstrapping(human_responses, counts, base=base)
        jsd_samples.append(jsd_val)

    jsd_samples = np.array(jsd_samples)

    if return_samples:
        return jsd_samples

    mean_jsd = jsd_samples.mean()
    ci_lower, ci_upper = np.percentile(jsd_samples, [2.5, 97.5])

    return (("mean", mean_jsd), ("ci_lower", ci_lower), ("ci_upper", ci_upper))


/var/folders/8s/b_t8p6s52r9c8k1j22zs0l740000gn/T/ipykernel_57581/2968426189.py:1: DtypeWarning: Columns (19,21,22,23,25,26,27,29,30,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  human_df = pd.read_csv("data/2020 ANES_test.csv")


In [439]:
#Create CIs (human vs conditions) and compare them
ci_replicate = []
ci_reformulated = []
ci_preamble = []
ci_priming = []
ci_reverse_coded = []

for qid in qids:
    ci_replicate.append((name_dict[qid], bootstrap_jsd(path = replicate_path, qid = qid)))
    ci_reformulated.append((name_dict[qid], bootstrap_jsd(path = reformulated_path, qid = qid)))
    ci_priming.append((name_dict[qid], bootstrap_jsd(path = priming_path, qid = qid)))
    ci_preamble.append((name_dict[qid], bootstrap_jsd(path = preamble_path, qid = qid)))

ci_replicate = [{item[0] : item[1]} for item in ci_replicate]
ci_reformulated = [{item[0] : item[1]} for item in ci_reformulated]
ci_priming = [{item[0] : item[1]} for item in ci_priming]
ci_preamble = [{item[0] : item[1]} for item in ci_preamble]

ci_replicate = {list(d.keys())[0]: list(d.values())[0] for d in ci_replicate}
ci_priming = {list(d.keys())[0]: list(d.values())[0] for d in ci_priming}
ci_reformulated = {list(d.keys())[0]: list(d.values())[0] for d in ci_reformulated}
ci_preamble = {list(d.keys())[0]: list(d.values())[0] for d in ci_preamble}


ci_reverse_coded = []

for qid in qids_reverse_coded:
    ci_reverse_coded.append((name_dict[qid], bootstrap_jsd_for_reverse_coded(path = reversed_path, qid = qid)))

ci_reverse_coded = [{item[0] : item[1]} for item in ci_reverse_coded]
ci_reverse_coded = {list(d.keys())[0]: list(d.values())[0] for d in ci_reverse_coded}

In [440]:
# Mean JS divergence and 95% bootstrap confidence intervals between ANSE and silicon distributions by item and condition.
def format_ci_data(ci_tuple):
    # Convert the list of tuples structure into a dictionary for easy access
    if ci_tuple is None:
        return "N/A"
    data = dict(ci_tuple)
    
    mean = data['mean']
    ci_lower = data['ci_lower']
    ci_upper = data['ci_upper']

    # Round and format
    mean_str = f"{mean:.4f}"
    lower_str = f"{ci_lower:4f}"
    upper_str = f"{ci_upper:.4f}"

    return f"{mean_str} [{lower_str}, {upper_str}]"

results_mapping = [
    ('Replicate', ci_replicate),
    ('Reformulated', ci_reformulated),
    ('Reverse-coded', ci_reverse_coded),
    ('Priming', ci_priming),
    ('Preamble', ci_preamble),
]
question_order = ['Climate Change','Current Economy','Refugee Allowing','Health Insurance','Income Inequality','Race Diversity','Drug Addiction','Gay Marriage','Gun Regulation','Gender Role']

ci_df = pd.DataFrame({'Question': question_order})
# Iterate over the mapping and add a column for each result set
for column_name, result_dict in results_mapping:
    # Extract data in the precise 'question_order' and apply the formatting function
    ordered_formatted_values = [
        format_ci_data(result_dict.get(q,None))
        for q in question_order
    ]

    ci_df[column_name] = ordered_formatted_values


In [111]:
#for table 2 in the appendix

# ci_comparison = {key: (ci_replicate[key], ci_reformulated[key], ci_preamble[key], ci_priming[key]) for key in ci_replicate.keys()}
# display(ci_comparison)
# display(ci_replicate)
# display(ci_reformulated)
# display(ci_reverse_coded)
# display(ci_preamble)
# display(ci_priming)


In [441]:
ci_df

,Question,Replicate,Reformulated,Reverse-coded,Priming,Preamble
0,Climate Change,"0.0578 [0.053730, 0.0622]","0.0261 [0.022759, 0.0296]","0.3404 [0.334297, 0.3467]","0.2109 [0.205346, 0.2166]","0.0807 [0.076111, 0.0857]"
1,Current Economy,"0.2324 [0.226797, 0.2378]","0.1725 [0.164224, 0.1808]",N/A,"0.2801 [0.276658, 0.2837]","0.2219 [0.216065, 0.2277]"
2,Refugee Allowing,"0.1134 [0.106930, 0.1202]","0.0269 [0.023391, 0.0304]","0.0489 [0.044114, 0.0537]","0.1095 [0.102735, 0.1161]","0.2490 [0.242947, 0.2546]"
3,Health Insurance,"0.0452 [0.040722, 0.0498]","0.0708 [0.065304, 0.0764]",N/A,"0.0794 [0.073470, 0.0853]","0.2073 [0.200515, 0.2143]"
4,Income Inequality,"0.0445 [0.039926, 0.0492]","0.1045 [0.098949, 0.1097]","0.0041 [0.002814, 0.0056]","0.0861 [0.080104, 0.0921]","0.0905 [0.084384, 0.0971]"
5,Race Diversity,"0.2540 [0.249017, 0.2589]","0.1863 [0.179702, 0.1928]","0.1141 [0.107440, 0.1213]","0.2674 [0.267372, 0.2674]","0.2479 [0.242138, 0.2531]"
6,Drug Addiction,"0.1660 [0.163640, 0.1682]","0.1441 [0.143907, 0.1444]",N/A,"0.1727 [0.172733, 0.1727]","0.1727 [0.172733, 0.1727]"
7,Gay Marriage,"0.0412 [0.037049, 0.0456]","0.0320 [0.028349, 0.0360]","0.0949 [0.094535, 0.0954]","0.0296 [0.025894, 0.0334]","0.0600 [0.054802, 0.0650]"
8,Gun Regulation,"0.0028 [0.001684, 0.0040]","0.0150 [0.012430, 0.0179]",N/A,"0.0134 [0.011011, 0.0159]","0.0129 [0.010463, 0.0156]"
9,Gender Role,"0.0770 [0.071204, 0.0830]","0.0103 [0.008116, 0.0127]","0.8471 [0.847077, 0.8471]","0.1823 [0.177230, 0.1875]","0.1400 [0.133655, 0.1463]"


In [442]:
file_path = f'tables/{model_name}-jsd-ci-t=0.csv'
ci_df.to_csv(file_path, index=False)